# PGDH Binder Design Visualization

Interactive exploration of BoltzGen-designed binders targeting 15-PGDH (PDB: 2GDZ).

**Prerequisites**: Download BoltzGen outputs from Lyceum storage first:
```bash
source .venv/bin/activate
mkdir -p pgdh_campaign/out/boltzgen
lyceum storage download output/boltzgen/final_ranked_designs/all_designs_metrics.csv \
  --output pgdh_campaign/out/boltzgen/all_designs_metrics.csv
lyceum storage download output/boltzgen/intermediate_designs_inverse_folded/aggregate_metrics_analyze.csv \
  --output pgdh_campaign/out/boltzgen/aggregate_metrics.csv
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

try:
    import py3Dmol
    HAS_3DMOL = True
except ImportError:
    HAS_3DMOL = False
    print("py3Dmol not installed. Install with: pip install py3Dmol")
    print("3D structure views will be skipped.")

OUT_DIR = Path("out/boltzgen")
STRUCTURES_DIR = Path("structures")

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Load Design Metrics

In [ ]:
# Try loading from the final ranked designs CSV first, fall back to aggregate
metrics_path = OUT_DIR / "all_designs_metrics.csv"
agg_path = OUT_DIR / "aggregate_metrics.csv"

if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    print(f"Loaded {len(df)} designs from {metrics_path}")
elif agg_path.exists():
    df = pd.read_csv(agg_path)
    print(f"Loaded {len(df)} designs from {agg_path}")
else:
    raise FileNotFoundError(
        f"No metrics CSV found. Expected one of:\n"
        f"  {metrics_path}\n  {agg_path}\n"
        f"Download from Lyceum storage first (see instructions above)."
    )

print(f"\nColumns: {list(df.columns)}")
df.head()

## 2. Design Quality Overview

In [ ]:
# Identify available metric columns
rmsd_cols = [c for c in df.columns if 'rmsd' in c.lower()]
frac_cols = [c for c in df.columns if 'fraction' in c.lower() or c.endswith('_fraction')]
score_cols = [c for c in df.columns if any(k in c.lower() for k in ['iptm', 'plddt', 'ptm'])]

print("RMSD columns:", rmsd_cols)
print("Fraction columns:", frac_cols)
print("Score columns:", score_cols)

# Summary statistics for key metrics
key_cols = rmsd_cols + score_cols
if key_cols:
    df[key_cols].describe().round(3)

In [ ]:
# Plot RMSD distributions
if rmsd_cols:
    fig, axes = plt.subplots(1, len(rmsd_cols), figsize=(6 * len(rmsd_cols), 5))
    if len(rmsd_cols) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, rmsd_cols):
        vals = df[col].dropna()
        ax.hist(vals, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
        ax.axvline(2.5, color='red', linestyle='--', label='threshold (2.5 Å)')
        ax.set_xlabel(f'{col} (Å)')
        ax.set_ylabel('Count')
        ax.set_title(f'{col} distribution')
        ax.legend()
        
        n_pass = (vals < 2.5).sum()
        ax.text(0.95, 0.95, f'{n_pass}/{len(vals)} pass',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.suptitle('Self-Consistency: Refolding RMSD', y=1.02, fontsize=14)
    plt.show()
else:
    print("No RMSD columns found in metrics.")

In [ ]:
# Amino acid composition
if frac_cols:
    fig, ax = plt.subplots(figsize=(14, 5))
    
    # Clean column names for display
    labels = [c.replace('_fraction', '').replace('_Fraction', '') for c in frac_cols]
    means = df[frac_cols].mean()
    stds = df[frac_cols].std()
    
    bars = ax.bar(labels, means, yerr=stds, capsize=3, color='steelblue', 
                  edgecolor='black', alpha=0.7)
    ax.axhline(0.3, color='red', linestyle='--', alpha=0.5, label='max threshold (0.3)')
    ax.set_ylabel('Fraction')
    ax.set_title('Amino Acid Composition of Designed Binders')
    ax.legend()
    
    # Highlight any that exceed threshold
    for i, (mean, label) in enumerate(zip(means, labels)):
        if mean > 0.3:
            bars[i].set_color('salmon')
    
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No amino acid fraction columns found.")

## 3. Filter Designs

Apply BoltzGen quality thresholds:

In [ ]:
# Apply filters
filtered = df.copy()
filter_log = []

# RMSD filters
for col in rmsd_cols:
    if col in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered[col] < 2.5]
        after = len(filtered)
        filter_log.append(f"{col} < 2.5 Å: {before} → {after} ({before - after} removed)")

# AA fraction filters
for col in frac_cols:
    if col in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered[col] < 0.3]
        after = len(filtered)
        if before != after:
            filter_log.append(f"{col} < 0.3: {before} → {after} ({before - after} removed)")

print(f"Designs after filtering: {len(filtered)} / {len(df)}")
print()
for log in filter_log:
    print(f"  {log}")

if len(filtered) > 0:
    print(f"\nTop designs:")
    display_cols = ['id'] if 'id' in filtered.columns else []
    display_cols += [c for c in rmsd_cols + score_cols if c in filtered.columns]
    if display_cols:
        filtered[display_cols].head(10)
    else:
        filtered.head(10)

## 4. 3D Structure Visualization

Download CIF files from Lyceum storage first:
```bash
mkdir -p pgdh_campaign/out/boltzgen/designs
# Download top designs
lyceum storage ls output/boltzgen/final_ranked_designs/final_30_designs/
# For each .cif file:
lyceum storage download output/boltzgen/final_ranked_designs/final_30_designs/rank1_*.cif \
  --output pgdh_campaign/out/boltzgen/designs/rank1.cif
```

In [ ]:
# Find downloaded CIF files
design_dirs = [
    OUT_DIR / "designs",
    OUT_DIR / "final_30_designs",
    OUT_DIR / "intermediate_ranked_10_designs",
]

cif_files = []
for d in design_dirs:
    if d.exists():
        cif_files.extend(sorted(d.glob("*.cif")))

# Also check for directly downloaded files
cif_files.extend(sorted(OUT_DIR.glob("*.cif")))

# Deduplicate
cif_files = list(dict.fromkeys(cif_files))

print(f"Found {len(cif_files)} CIF files:")
for f in cif_files:
    print(f"  {f} ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
def view_complex(cif_path, width=800, height=600):
    """Visualize a binder-target complex from a CIF file using py3Dmol."""
    if not HAS_3DMOL:
        print("py3Dmol not available. Skipping 3D view.")
        return None
    
    cif_text = Path(cif_path).read_text()
    
    view = py3Dmol.view(width=width, height=height)
    view.addModel(cif_text, 'cif')
    
    # Color by chain: target (A) in gray, binder (B) in cyan
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': '#999999', 'opacity': 0.8}})
    view.setStyle({'chain': 'B'}, {'cartoon': {'color': '#00CED1'}})
    
    # Show interface residues as sticks
    view.addStyle(
        {'chain': 'B', 'within': {'distance': 5, 'sel': {'chain': 'A'}}},
        {'stick': {'color': '#00CED1'}}
    )
    view.addStyle(
        {'chain': 'A', 'within': {'distance': 5, 'sel': {'chain': 'B'}}},
        {'stick': {'color': '#FF6347'}}
    )
    
    view.zoomTo()
    view.setBackgroundColor('white')
    
    print(f"Viewing: {Path(cif_path).name}")
    print(f"  Gray = PGDH target (chain A)")
    print(f"  Cyan = Designed binder (chain B)")
    print(f"  Sticks = Interface residues (< 5 Å)")
    
    return view.show()

In [ ]:
# View the top-ranked design
if cif_files:
    view_complex(cif_files[0])
else:
    print("No CIF files found. Download designs from Lyceum storage first.")
    print("See instructions in the markdown cell above.")

In [ ]:
# View all downloaded designs
for cif in cif_files[1:]:
    view_complex(cif)

## 5. Target Structure Reference

View the PGDH target (2GDZ) with active site and dimer interface residues highlighted.

In [ ]:
target_cif = STRUCTURES_DIR / "2GDZ.cif"
target_pdb = STRUCTURES_DIR / "2GDZ.pdb"

target_path = target_cif if target_cif.exists() else target_pdb if target_pdb.exists() else None

if target_path and HAS_3DMOL:
    fmt = 'cif' if target_path.suffix == '.cif' else 'pdb'
    target_text = target_path.read_text()
    
    view = py3Dmol.view(width=800, height=600)
    view.addModel(target_text, fmt)
    
    # Base style: gray cartoon
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': '#CCCCCC', 'opacity': 0.7}})
    
    # Active site residues (auth_seq_id): Ser138, Gln148, Tyr151, Lys155, Phe185, Tyr217
    active_site = [138, 148, 151, 155, 185, 217]
    view.addStyle(
        {'chain': 'A', 'resi': active_site},
        {'stick': {'color': '#FF4500'}, 'cartoon': {'color': '#FF4500'}}
    )
    
    # Dimer interface residues: Ala146, Ala153, Phe161, Leu167, Ala168, Leu171, Met172, Tyr206
    dimer_interface = [146, 153, 161, 167, 168, 171, 172, 206]
    view.addStyle(
        {'chain': 'A', 'resi': dimer_interface},
        {'stick': {'color': '#1E90FF'}, 'cartoon': {'color': '#1E90FF'}}
    )
    
    view.zoomTo()
    view.setBackgroundColor('white')
    
    print("15-PGDH (2GDZ) — Chain A")
    print(f"  Red/Orange = Active site: {active_site}")
    print(f"  Blue = Dimer interface: {dimer_interface}")
    view.show()
elif not HAS_3DMOL:
    print("py3Dmol not available.")
else:
    print(f"Target structure not found at {target_cif} or {target_pdb}")

## 6. Extract Binder Sequences

In [ ]:
def extract_sequence_from_cif(cif_path, chain_id='B'):
    """Extract protein sequence for a chain from a CIF file."""
    three_to_one = {
        'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
        'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
        'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
        'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V',
    }
    
    residues = {}
    with open(cif_path) as f:
        in_atom_site = False
        headers = []
        for line in f:
            line = line.strip()
            if line.startswith('_atom_site.'):
                in_atom_site = True
                headers.append(line.split('.')[1])
                continue
            if in_atom_site and not line.startswith('_') and not line.startswith('#') and line:
                fields = line.split()
                if len(fields) < len(headers):
                    in_atom_site = False
                    continue
                row = dict(zip(headers, fields))
                
                chain = row.get('label_asym_id', row.get('auth_asym_id', ''))
                if chain != chain_id:
                    continue
                if row.get('group_PDB') != 'ATOM':
                    continue
                    
                resname = row.get('label_comp_id', '')
                resnum = int(row.get('label_seq_id', 0))
                if resname in three_to_one and resnum not in residues:
                    residues[resnum] = three_to_one[resname]
            elif in_atom_site and (line.startswith('#') or line.startswith('_') or line == ''):
                if residues:  # Already got data, stop
                    break
    
    if not residues:
        return ""
    return ''.join(residues[k] for k in sorted(residues))


# Extract sequences from all design CIFs
sequences = {}
for cif in cif_files:
    seq = extract_sequence_from_cif(cif, chain_id='B')
    if not seq:  # Try chain C (some configs use different IDs)
        seq = extract_sequence_from_cif(cif, chain_id='C')
    if seq:
        sequences[cif.stem] = seq
        print(f"{cif.stem}: {len(seq)} AA")
        print(f"  {seq[:60]}{'...' if len(seq) > 60 else ''}")
    else:
        print(f"{cif.stem}: could not extract binder sequence")

print(f"\nExtracted {len(sequences)} binder sequences")

In [ ]:
# Save as FASTA
if sequences:
    fasta_path = OUT_DIR / "binder_sequences.fasta"
    with open(fasta_path, 'w') as f:
        for name, seq in sequences.items():
            f.write(f">{name}\n{seq}\n")
    print(f"Saved {len(sequences)} sequences to {fasta_path}")
else:
    print("No sequences to save.")

## 7. Summary

In [ ]:
print("=" * 60)
print("PGDH Binder Design Campaign Summary")
print("=" * 60)
print(f"Total designs generated: {len(df)}")
print(f"Designs passing filters: {len(filtered)}")
print(f"Structure files downloaded: {len(cif_files)}")
print(f"Sequences extracted: {len(sequences)}")
print()
if len(filtered) > 0 and rmsd_cols:
    for col in rmsd_cols:
        if col in filtered.columns:
            print(f"Best {col}: {filtered[col].min():.3f} Å")
print()
print("Next steps:")
print("  1. Cross-validate top designs with Boltz-2 (/boltz)")
print("  2. Score with ipSAE (/pgdh_ipsae)")
print("  3. Select final 10 for submission")